# Model explorer: training, evaluation, and inference

Choose a **selected Tanh, tuned Snake, or spectral + Adam Fourier recipe**, train
it, inspect the results, and explore predictions on new input intervals.

**Start here:** complete the [README setup](../README.md#setup), then select the
repository's `.venv/bin/python` notebook kernel. A notebook interface and
`ipykernel` are optional environment tools needed for interactive execution.

Run cells from top to bottom. The default **10,000-point** budget matches the
main experiments; change it to **1,000** to explore fewer observations. Each
training call saves a fresh timestamped run with its exact configuration.
The output figures and reference scores concern one noise-free periodic target.

### Workflow
1. Select a recipe and sample budget.
2. Inspect disjoint data splits and optional Fourier initialization.
3. Train using the existing project runner.
4. Inspect validation, interpolation, and extrapolation metrics.
5. Reload the selected checkpoint and predict without retraining.

These are selected recipes within the tested configurations, rather than
universally best models. Tanh and Snake have different preprocessing and
training settings; their tuned comparison is not an activation-only experiment.


## 1. Load the project

The repository is detected automatically from the notebook's working directory.
No machine-specific paths are required. If detection fails, start your notebook
interface from the repository root.


In [1]:
from pathlib import Path
import os
import sys

# ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
#              if (p / "pyproject.toml").is_file()
#              and (p / "function_approximation").is_dir()), None)
# if ROOT is None:
#     raise RuntimeError("Start the notebook from the repository or notebooks/ directory.")
# sys.path.insert(0, str(ROOT))
# os.environ.setdefault("MPLCONFIGDIR", str(ROOT / "outputs/.matplotlib"))

from datetime import datetime, timezone
import json
import numpy as np
import pandas as pd
import torch
import yaml
import matplotlib.pyplot as plt
from IPython.display import Image, display
from function_approximation.data.generation import DataGenerator
from function_approximation.data.preprocessing import DataPreprocessor
from function_approximation.models import build_model
from function_approximation.training.config import load_experiment_config
from function_approximation.inference import load_predictor
from function_approximation.evaluation.metrics import compute_metrics
from function_approximation.evaluation.visualization import (
    GROUND_TRUTH, PREDICTION, pi_axis, style_figure,
)
from scripts.run_experiments import run_experiment, set_seed

torch.set_num_threads(1)
# print("Python:", sys.executable)
# print("PyTorch:", torch.__version__)
# print("Repository:", ROOT)


## 2. Select the model and sample budget

Set `MODEL` to **"fourier"**, **"tanh"**, or **"snake"** before running the cells.
Each choice retains its own preprocessing, learning rate, batch size, and stopping settings.

| Recipe | Inputs | Architecture | LR | Batch | Max epochs / patience |
|---|---|---|---|---|---|
| Tanh | Standardized | [64, 256, 128] | 0.0001 | 64 | 20,000 / 150 |
| Snake | Raw | [64, 256, 128], trainable a | 0.001 for weights and a | 128 | 8,000 / 1,500 |
| Fourier | Raw | K=8, spectral + Adam | 0.001 | 64 | 30,000 / 3,000 |

The Tanh settings are loaded from the `learning_rate_low` ablation variant.
Snake and Fourier use their standalone experiment recipes. The default seed is 42.
A smaller sample budget also means fewer mini-batch optimizer updates per epoch;
this notebook does not retune settings for each sample budget.


In [ ]:
MODEL = "fourier"  # Choose "fourier", "tanh", or "snake".
SAMPLE_SIZE = 10_000  # Or 1_000 for a reduced-data experiment.
SEED = 42

recipe_paths = {
    "fourier": "configs/experiments/fourier_rich_periodic.yaml",
    "tanh": "configs/experiments/mlp_rich_periodic.yaml",
    "snake": "configs/experiments/snake_rich_periodic.yaml",
}
if MODEL not in recipe_paths:
    raise ValueError(f"MODEL must be one of {list(recipe_paths)}")
config = load_experiment_config(ROOT / recipe_paths[MODEL])
if MODEL == "tanh":
    study = yaml.safe_load((ROOT / "configs/ablations/mlp/architecture.yaml").read_text())
    config["model"].update(study["model"])
    for section, values in study["variants"]["learning_rate_low"].items():
        config[section].update(values)
    config["training"]["max_epochs"] = study["max_epochs"]
config["data"]["sample_size"] = SAMPLE_SIZE
config["experiment"]["seed"] = SEED
config["experiment"]["run_name"] = f"explorer_{MODEL}_n{SAMPLE_SIZE}_seed{SEED}"
config["tracking"]["enabled"] = False
session_dir = ROOT / "outputs/notebooks/model_explorer" / datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
session_dir.mkdir(parents=True, exist_ok=False)
config["experiment"]["output_dir"] = (session_dir / MODEL / "runs").relative_to(ROOT).as_posix()
config_path = session_dir / "config.yaml"
config_path.write_text(yaml.safe_dump(config, sort_keys=False))
config = load_experiment_config(config_path)
display(pd.Series({
    "recipe": MODEL, "sample_size": SAMPLE_SIZE, "seed": SEED,
    "model": config["model"], "scaling": config["preprocessing"]["scaling"],
    "learning_rate": config["training"]["learning_rate"],
    "batch_size": config["training"]["batch_size"],
    "max_epochs": config["training"]["max_epochs"],
    "patience": config["training"]["patience"],
    "output_dir": config["experiment"]["output_dir"],
}, name="Selected experiment"))


## 3. Generate and inspect the observations

Training inputs span **[−10π, 10π]**. Validation and test are disjoint interior
grids. Near **(10π, 20π]** and far **(50π, 60π]** data are evaluation-only.
The reference function generates labels; it is never called by model inference.


In [ ]:
generator = DataGenerator(**config["data"])
dataset = generator.generate()
bands = generator.generate_extrapolation()
display(pd.DataFrame([
    {"split": name, "samples": len(x), "min_x / pi": float(np.min(x) / np.pi),
     "max_x / pi": float(np.max(x) / np.pi)}
    for name, (x, y) in {**dataset, **bands}.items()
]))
for left, right in (("train", "val"), ("train", "test"), ("val", "test")):
    assert np.intersect1d(dataset[left][0].astype(np.float32),
                          dataset[right][0].astype(np.float32)).size == 0
print("Verified: interpolation splits are disjoint in raw float32 coordinates.")


## 4. Inspect initialization (Fourier only)

The Fourier model predicts

\[
\hat f(x)=b+\sum_{k=1}^{K}\left[a_k\sin(\omega_kx)+c_k\cos(\omega_kx)\right].
\]

The initializer uses **training data only**:
1. Subtract the label mean to obtain a residual.
2. Score 4,096 candidate frequencies with Lomb–Scargle.
3. Refine the strongest available candidate locally.
4. Jointly refit selected sine/cosine weights and bias by least squares.
5. Update the residual and repeat, up to K components.

This cell previews the starting fit. The training runner repeats the same
initialization with the same seed/data so artifact handling stays in the existing pipeline.
The displayed float64 least-squares result can differ slightly from float32 model evaluation.

For Tanh and Snake this cell prints the architecture and skips spectral fitting.


In [ ]:
if config["model"]["type"] == "fourier":
    preprocessor = DataPreprocessor(**config["preprocessing"])
    processed = preprocessor.fit_transform_dataset(dataset)
    set_seed(SEED)
    architecture = {k: v for k, v in config["model"].items() if k != "type"}
    preview_model = build_model(config["model"]["type"], architecture)
    initialization = preview_model.initialize_from_data(*processed["train"])
    display(pd.DataFrame(initialization["spectrum"]["components"]))
    with torch.inference_mode():
        initial_prediction = preview_model(torch.from_numpy(processed["train"][0])).numpy()
    print("Active starting components:", initialization["active_components"])
    print("Starting model train RMSE:", compute_metrics(dataset["train"][1], initial_prediction)["rmse"])
else:
    print("Selected architecture:", config["model"])
    print("This recipe uses random network initialization; spectral fitting is skipped.")


## 5. Train and save the validation-selected model

This cell calls the shared experiment runner with the chosen recipe. Preprocessing
fits training inputs only; validation MSE selects the checkpoint. Test and
extrapolation scores do not enter training or checkpoint selection.

Each execution trains again and creates a new run. To switch models, change
`MODEL` and rerun from section 2. Training time depends on the recipe and machine;
the inference section can be rerun independently after a checkpoint is loaded.


In [ ]:
run_dir = run_experiment(config_path)
print("Run root:", run_dir)


## 6. Read the results

**Lower RMSE is better.** `test` measures held-out interpolation; `extrap_near`
and `extrap_far` measure extrapolation. Compare these with the training-mean baseline.
The figures show training history and true-versus-predicted curves.

This notebook evaluates one recipe. Use the README's ablation commands to compare
initialization strategies, component counts, or multiple seeds.


In [ ]:
metrics = json.loads((run_dir / "metrics/metrics.json").read_text())
baseline = json.loads((run_dir / "metrics/baseline_metrics.json").read_text())
score_table = pd.DataFrame(metrics).T
score_table["mean_baseline_rmse"] = pd.Series({k: v["rmse"] for k, v in baseline["metrics"].items()})
display(score_table)
for plot_name in ("learning_curves.png", "extrapolation.png"):
    plot_file = run_dir / "plots" / plot_name
    if plot_file.exists():
        display(Image(filename=str(plot_file)))


## 7. Reload the checkpoint

Inference requires only the saved model and matching preprocessor. Supply the
**run root**, not its `model/` folder. To explore an earlier run, replace
`INFERENCE_RUN` with that run's path; this cell does not retrain.


In [ ]:
INFERENCE_RUN = run_dir  # Or ROOT / "outputs/.../timestamp" for an existing run.
predictor = load_predictor(INFERENCE_RUN)
inference_config = yaml.safe_load((Path(INFERENCE_RUN) / "config.yaml").read_text())
print("Loaded:", INFERENCE_RUN)
print("Saved model:", inference_config["model"])
if inference_config["model"]["type"] == "fourier":
    display(pd.DataFrame(predictor.model.spectrum()["components"]))


## 8. Explore predictions on new inputs

Edit the interval below and rerun **this cell only** to explore extrapolation.
The default **[12π, 18π]** is outside the training interval. Try **[50π, 60π]**
or a more distant interval.

`predictor.predict()` uses no labels or target calls. We compute reference values
**separately**, only to plot and score the predictions. This diagnostic interval
does not affect the already selected model. Curves may visually overlap despite
small nonzero error.


In [ ]:
START_PI = 12.0
END_PI = 18.0
N_POINTS = 2000
assert END_PI > START_PI and N_POINTS >= 2

x_new = np.linspace(START_PI * np.pi, END_PI * np.pi, N_POINTS)
y_pred = predictor.predict(x_new)
reference = DataGenerator(**inference_config["data"])
y_true = reference.target_fn(x_new)  # Evaluation only, separate from inference.
scores = compute_metrics(y_true, y_pred)
comparison = pd.DataFrame({"x / pi": x_new / np.pi, "true value": y_true,
                           "prediction": y_pred, "absolute error": np.abs(y_true - y_pred)})
display(comparison.head(10))
print(f"Interval RMSE: {scores['rmse']:.6e}")

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(x_new, y_true, color=GROUND_TRUTH, linewidth=4, label="True value")
ax.plot(x_new, y_pred, color=PREDICTION, linewidth=2, linestyle="--", label="Model prediction")
ax.set(title=f"Inference on new inputs · RMSE {scores['rmse']:.3e}", xlabel="Input x", ylabel="f(x)")
pi_axis(ax, x_new.min(), x_new.max(), ticks=5)
ax.legend(frameon=False)
style_figure(fig)
fig.tight_layout()
plot_path = Path(INFERENCE_RUN) / "plots" / "inference_comparison.png"
fig.savefig(plot_path, dpi=250, bbox_inches="tight")
fig.savefig(plot_path.with_suffix(".svg"), bbox_inches="tight")
comparison.to_csv(Path(INFERENCE_RUN) / "data/inference_comparison.csv", index=False)
plt.close(fig)
display(Image(filename=str(plot_path)))
print("Inference comparison saved under:", Path(INFERENCE_RUN))


## 9. Check saved-model consistency

Reloaded predictions should exactly match the original prediction exports for
every split. This verifies checkpoint/preprocessor consistency; it is not a
separate proof of generalization or absence of leakage.


In [ ]:
saved_metrics = json.loads((Path(INFERENCE_RUN) / "metrics/metrics.json").read_text())
with np.load(Path(INFERENCE_RUN) / "data/predictions.npz") as saved:
    for split in saved_metrics:
        np.testing.assert_array_equal(predictor.predict(saved[f"{split}_x"]), saved[f"{split}_pred"])
    checked = compute_metrics(saved["test_y"], predictor.predict(saved["test_x"]))
assert np.isclose(checked["rmse"], saved_metrics["test"]["rmse"], rtol=1e-10, atol=1e-12)
print("Verified: all split predictions match the saved checkpoint.")
print("Verified test RMSE:", checked["rmse"])


## Where to find your outputs

| Path inside the run | Contents |
|---|---|
| `config.yaml` | Resolved training settings |
| `model/` | Selected checkpoint and fitted preprocessor |
| `metrics/` | Per-split scores and mean baseline |
| `diagnostics/` | History, timing, initialization report, fitted spectrum |
| `plots/` | Training/evaluation figures and your latest inference comparison |
| `data/` | Prediction exports and your latest inference comparison CSV |

The training run is immutable with respect to new training executions; inference
comparison files are replaced when you explore a new interval in the same run.

For controlled comparisons and interpretation, see the [README](../README.md).
Finite sample coverage, frequency mismatch, and numerical precision limit distant
extrapolation even when near-band predictions look excellent.
